# MLB Hitter Breakout Predictor — Data Collection

## Research Question

Can plate discipline and quality of contact help predict which MLB hitters will improve or break out in the following season?

### Plate Discipline
- Chase%
- Whiff%
- Walk%
- Strikeout%

### Contact Quality
- Hard-Hit%
- Barrel%
- Average Exit Velocity
- Maximum Exit Velocity
- Sweet-Spot%

### Target
Future offensive performance, measured primarily using wOBA.

In [1]:
from pathlib import Path
from io import StringIO

import pandas as pd
import numpy as np
import requests


from pybaseball import (
    statcast_batter_exitvelo_barrels,
    statcast_batter_percentile_ranks
)

In [2]:
discipline_2025 = statcast_batter_percentile_ranks(2025)

print("Rows:", discipline_2025.shape[0])
print("Columns:", discipline_2025.shape[1])

discipline_2025.columns.tolist()

Rows: 617
Columns: 23


['player_name',
 'player_id',
 'year',
 'xwoba',
 'xba',
 'xslg',
 'xiso',
 'xobp',
 'brl',
 'brl_percent',
 'exit_velocity',
 'max_ev',
 'hard_hit_percent',
 'k_percent',
 'bb_percent',
 'whiff_percent',
 'chase_percent',
 'arm_strength',
 'sprint_speed',
 'oaa',
 'bat_speed',
 'squared_up_rate',
 'swing_length']

In [3]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DATA_DIR)
print("Processed data folder:", PROCESSED_DATA_DIR)

Project root: /Users/darekstaudt/Desktop/MLB-breakout-predictor
Raw data folder: /Users/darekstaudt/Desktop/MLB-breakout-predictor/data/raw
Processed data folder: /Users/darekstaudt/Desktop/MLB-breakout-predictor/data/processed


In [4]:
SEASON = 2025
MIN_BBE = 100

contact_2025 = statcast_batter_exitvelo_barrels(
    SEASON,
    MIN_BBE
)

contact_2025.head()

,"last_name, first_name",player_id,attempts,avg_hit_angle,anglesweetspotpercent,max_hit_speed,avg_hit_speed,ev50,fbld,gb,max_distance,avg_distance,avg_hr_distance,ev95plus,ev95percent,barrels,brl_percent,brl_pa
0,"Arraez, Luis",650333,618,12.2,36.9,107.8,86.1,93.6,87.6,85.1,402,152,370.0,103,16.7,7,1.1,1.0
1,"Kwan, Steven",680757,575,15.6,39.3,104.2,86.2,94.4,87.8,85.1,431,167,381.0,111,19.3,11,1.9,1.6
2,"Hoerner, Nico",663538,554,12.5,36.1,108.1,86.7,96.7,89.3,85.8,412,155,379.0,168,30.3,13,2.3,2.0
3,"Perdomo, Geraldo",672695,533,15.5,36.0,108.2,87.6,97.4,90.3,86.6,423,165,393.0,170,31.9,33,6.2,4.6
4,"Betts, Mookie",605141,531,18.0,37.7,108.4,89.1,98.1,91.6,86.0,412,191,390.0,190,35.8,29,5.5,4.4


In [5]:
print("Rows:", contact_2025.shape[0])
print("Columns:", contact_2025.shape[1])

contact_2025.columns.tolist()

Rows: 389
Columns: 18


['last_name, first_name',
 'player_id',
 'attempts',
 'avg_hit_angle',
 'anglesweetspotpercent',
 'max_hit_speed',
 'avg_hit_speed',
 'ev50',
 'fbld',
 'gb',
 'max_distance',
 'avg_distance',
 'avg_hr_distance',
 'ev95plus',
 'ev95percent',
 'barrels',
 'brl_percent',
 'brl_pa']

In [6]:
contact_2025["season"] = 2025

contact_file = RAW_DATA_DIR / "statcast_contact_2025.csv"

contact_2025.to_csv(
    contact_file,
    index=False
)

print(f"Saved raw contact data to: {contact_file}")

Saved raw contact data to: /Users/darekstaudt/Desktop/MLB-breakout-predictor/data/raw/statcast_contact_2025.csv


In [7]:
contact_columns = [
    "player_id",
    "last_name, first_name",
    "season",
    "attempts",
    "avg_hit_speed",
    "max_hit_speed",
    "ev95percent",
    "brl_percent",
    "brl_pa",
    "avg_hit_angle",
    "anglesweetspotpercent"
]

contact_clean_2025 = contact_2025[contact_columns].copy()

contact_clean_2025.head()

,player_id,"last_name, first_name",season,attempts,avg_hit_speed,max_hit_speed,ev95percent,brl_percent,brl_pa,avg_hit_angle,anglesweetspotpercent
0,650333,"Arraez, Luis",2025,618,86.1,107.8,16.7,1.1,1.0,12.2,36.9
1,680757,"Kwan, Steven",2025,575,86.2,104.2,19.3,1.9,1.6,15.6,39.3
2,663538,"Hoerner, Nico",2025,554,86.7,108.1,30.3,2.3,2.0,12.5,36.1
3,672695,"Perdomo, Geraldo",2025,533,87.6,108.2,31.9,6.2,4.6,15.5,36.0
4,605141,"Betts, Mookie",2025,531,89.1,108.4,35.8,5.5,4.4,18.0,37.7


In [8]:
contact_clean_2025 = contact_clean_2025.rename(
    columns={
        "last_name, first_name": "player_name",
        "attempts": "bbe",
        "avg_hit_speed": "avg_exit_velocity",
        "max_hit_speed": "max_exit_velocity",
        "ev95percent": "hard_hit_pct",
        "brl_percent": "barrel_pct",
        "brl_pa": "barrel_per_pa",
        "avg_hit_angle": "avg_launch_angle",
        "anglesweetspotpercent": "sweet_spot_pct"
    }
)

contact_clean_2025.head()

,player_id,player_name,season,bbe,avg_exit_velocity,max_exit_velocity,hard_hit_pct,barrel_pct,barrel_per_pa,avg_launch_angle,sweet_spot_pct
0,650333,"Arraez, Luis",2025,618,86.1,107.8,16.7,1.1,1.0,12.2,36.9
1,680757,"Kwan, Steven",2025,575,86.2,104.2,19.3,1.9,1.6,15.6,39.3
2,663538,"Hoerner, Nico",2025,554,86.7,108.1,30.3,2.3,2.0,12.5,36.1
3,672695,"Perdomo, Geraldo",2025,533,87.6,108.2,31.9,6.2,4.6,15.5,36.0
4,605141,"Betts, Mookie",2025,531,89.1,108.4,35.8,5.5,4.4,18.0,37.7


In [9]:
CUSTOM_STATS = [
    "pa",
    "k_percent",
    "bb_percent",
    "woba",
    "xwoba",
    "whiff_percent",
    "oz_swing_percent",
    "swing_percent",
    "exit_velocity_avg",
    "sweet_spot_percent",
    "barrel_batted_rate",
    "hard_hit_percent",
]

selections = ",".join(CUSTOM_STATS)

url = (
    "https://baseballsavant.mlb.com/leaderboard/custom"
    f"?year=2025"
    "&type=batter"
    "&filter="
    "&min=100"
    f"&selections={selections}"
    "&chart=false"
    "&x=pa"
    "&y=pa"
    "&r=no"
    "&chartType=beeswarm"
    "&sort=1"
    "&sortDir=desc"
    "&csv=true"
)

response = requests.get(url)
response.raise_for_status()

discipline_raw_2025 = pd.read_csv(StringIO(response.text))

print("Rows:", discipline_raw_2025.shape[0])
print("Columns:", discipline_raw_2025.shape[1])

discipline_raw_2025.head()

discipline_raw_2025.columns.tolist()

Rows: 461
Columns: 15


['last_name, first_name',
 'player_id',
 'year',
 'pa',
 'k_percent',
 'bb_percent',
 'woba',
 'xwoba',
 'whiff_percent',
 'oz_swing_percent',
 'swing_percent',
 'exit_velocity_avg',
 'sweet_spot_percent',
 'barrel_batted_rate',
 'hard_hit_percent']

In [10]:
discipline_raw_2025[
    [
        "last_name, first_name",
        "player_id",
        "pa",
        "k_percent",
        "bb_percent",
        "whiff_percent",
        "oz_swing_percent",
        "woba",
        "xwoba",
    ]
].head(10)

,"last_name, first_name",player_id,pa,k_percent,bb_percent,whiff_percent,oz_swing_percent,woba,xwoba
0,"Kurtz, Nick",701762,489,30.9,12.9,35.5,22.4,0.419,0.372
1,"McMahon, Ryan",641857,586,32.3,11.9,35.2,25.1,0.306,0.322
2,"Vargas, Miguel",678246,569,17.6,9.8,20.6,21.5,0.314,0.319
3,"Keaschall, Luke",807712,207,14.0,9.2,18.5,20.0,0.363,0.324
4,"Beck, Jordan",687597,588,29.6,7.3,31.2,27.4,0.319,0.302
5,"Walls, Taylor",670764,317,22.1,8.2,20.3,21.2,0.265,0.261
6,"Pérez, Wenceel",672761,383,22.7,8.1,26.0,32.7,0.318,0.310
7,"Albies, Ozzie",645277,667,14.1,8.2,20.8,33.5,0.295,0.299
8,"Edman, Tommy",669242,377,16.2,5.0,19.8,28.4,0.284,0.321
9,"Yelich, Christian",592885,644,25.9,9.9,29.2,25.7,0.343,0.330


In [11]:
discipline_file = RAW_DATA_DIR / "savant_discipline_2025.csv"

discipline_raw_2025.to_csv(
    discipline_file,
    index=False
)

print(f"Saved raw discipline data to: {discipline_file}")

Saved raw discipline data to: /Users/darekstaudt/Desktop/MLB-breakout-predictor/data/raw/savant_discipline_2025.csv


In [12]:
discipline_columns = [
    "player_id",
    "last_name, first_name",
    "pa",
    "k_percent",
    "bb_percent",
    "whiff_percent",
    "oz_swing_percent",
    "woba",
    "xwoba"
]

discipline_clean_2025 = discipline_raw_2025[
    discipline_columns
].copy()

discipline_clean_2025["season"] = 2025

discipline_clean_2025.head()

discipline_clean_2025 = discipline_clean_2025.rename(
    columns={
        "last_name, first_name": "player_name",
        "pa": "plate_appearances",
        "k_percent": "strikeout_pct",
        "bb_percent": "walk_pct",
        "whiff_percent": "whiff_pct",
        "oz_swing_percent": "chase_pct"
    }
)

discipline_clean_2025.head()

,player_id,player_name,plate_appearances,strikeout_pct,walk_pct,whiff_pct,chase_pct,woba,xwoba,season
0,701762,"Kurtz, Nick",489,30.9,12.9,35.5,22.4,0.419,0.372,2025
1,641857,"McMahon, Ryan",586,32.3,11.9,35.2,25.1,0.306,0.322,2025
2,678246,"Vargas, Miguel",569,17.6,9.8,20.6,21.5,0.314,0.319,2025
3,807712,"Keaschall, Luke",207,14.0,9.2,18.5,20.0,0.363,0.324,2025
4,687597,"Beck, Jordan",588,29.6,7.3,31.2,27.4,0.319,0.302,2025


In [13]:
#Checking for Duplicates
print(
    "Contact duplicate player IDs:",
    contact_clean_2025["player_id"].duplicated().sum()
)

print(
    "Discipline duplicate player IDs:",
    discipline_clean_2025["player_id"].duplicated().sum()
)

Contact duplicate player IDs: 0
Discipline duplicate player IDs: 0


In [14]:
#Only keeping players who exist in both datasets
hitter_2025 = contact_clean_2025.merge(
    discipline_clean_2025,
    on=["player_id", "season"],
    how="inner",
    suffixes=("_contact", "_discipline")
)

print("Rows after merge:", hitter_2025.shape[0])
print("Columns after merge:", hitter_2025.shape[1])

hitter_2025.head()

#Checking if names match after merge
name_mismatches = hitter_2025[
    hitter_2025["player_name_contact"]
    != hitter_2025["player_name_discipline"]
]

print("Name mismatches:", len(name_mismatches))

name_mismatches[
    [
        "player_id",
        "player_name_contact",
        "player_name_discipline"
    ]
].head(20)

#Final Player column
hitter_2025["player_name"] = hitter_2025[
    "player_name_contact"
]

hitter_2025 = hitter_2025.drop(
    columns=[
        "player_name_contact",
        "player_name_discipline"
    ]
)

hitter_2025.head()

column_order = [
    "player_id",
    "player_name",
    "season",
    "plate_appearances",
    "bbe",

    # Plate discipline
    "walk_pct",
    "strikeout_pct",
    "whiff_pct",
    "chase_pct",

    # Contact quality
    "avg_exit_velocity",
    "max_exit_velocity",
    "hard_hit_pct",
    "barrel_pct",
    "barrel_per_pa",
    "sweet_spot_pct",
    "avg_launch_angle",

    # Performance
    "woba",
    "xwoba"
]

hitter_2025 = hitter_2025[column_order]

hitter_2025.head(10)

hitter_2025["xwoba_gap"] = (
    hitter_2025["xwoba"] - hitter_2025["woba"]
)

#xwoba gap potential breakout signals

hitter_2025[
    [
        "player_name",
        "woba",
        "xwoba",
        "xwoba_gap"
    ]
].sort_values(
    "xwoba_gap",
    ascending=False
).head(10)


# Saved merged 2025 dataset
processed_file = (
    PROCESSED_DATA_DIR
    / "hitter_features_2025.csv"
)

hitter_2025.to_csv(
    processed_file,
    index=False
)

print(f"Saved processed dataset to: {processed_file}")

Rows after merge: 389
Columns after merge: 19
Name mismatches: 0
Saved processed dataset to: /Users/darekstaudt/Desktop/MLB-breakout-predictor/data/processed/hitter_features_2025.csv


In [15]:
print("Rows after merge:", hitter_2025.shape[0])

Rows after merge: 389


## Historical Data Collection

Now that the 2025 workflow has been validated, the same process will be automated across multiple MLB seasons. A reusable function will:

1. Download Statcast contact-quality data.
2. Download Baseball Savant plate-discipline data.
3. Clean and standardize both datasets.
4. Merge them using MLB player ID.
5. Engineer additional features.
6. Save each season as a processed dataset.

In [16]:
CUSTOM_STATS = [
    "pa",
    "k_percent",
    "bb_percent",
    "woba",
    "xwoba",
    "whiff_percent",
    "oz_swing_percent",
    "swing_percent",
    "exit_velocity_avg",
    "sweet_spot_percent",
    "barrel_batted_rate",
    "hard_hit_percent",
]

def collect_hitter_season(season, min_bbe=100, min_pa=100):
    """
    Collect, clean, merge, and return MLB hitter data
    for one season.

    Parameters
    ----------
    season : int
        MLB season to collect.
    min_bbe : int
        Minimum number of batted-ball events required
        for the Statcast contact dataset.
    min_pa : int
        Minimum number of plate appearances required
        for the Baseball Savant custom leaderboard.

    Returns
    -------
    pandas.DataFrame
        One cleaned row per hitter for the selected season.
    """

    print(f"Collecting {season} data...")

    # -------------------------------
    # 1. CONTACT QUALITY DATA
    # -------------------------------

    contact = statcast_batter_exitvelo_barrels(
        season,
        min_bbe
    )

    contact["season"] = season

    contact_columns = [
        "player_id",
        "last_name, first_name",
        "season",
        "attempts",
        "avg_hit_speed",
        "max_hit_speed",
        "ev95percent",
        "brl_percent",
        "brl_pa",
        "avg_hit_angle",
        "anglesweetspotpercent",
    ]

    contact = contact[contact_columns].copy()

    contact = contact.rename(
        columns={
            "last_name, first_name": "player_name",
            "attempts": "bbe",
            "avg_hit_speed": "avg_exit_velocity",
            "max_hit_speed": "max_exit_velocity",
            "ev95percent": "hard_hit_pct",
            "brl_percent": "barrel_pct",
            "brl_pa": "barrel_per_pa",
            "avg_hit_angle": "avg_launch_angle",
            "anglesweetspotpercent": "sweet_spot_pct",
        }
    )

    # -------------------------------
    # 2. PLATE DISCIPLINE DATA
    # -------------------------------

    selections = ",".join(CUSTOM_STATS)

    url = (
        "https://baseballsavant.mlb.com/leaderboard/custom"
        f"?year={season}"
        "&type=batter"
        "&filter="
        f"&min={min_pa}"
        f"&selections={selections}"
        "&chart=false"
        "&x=pa"
        "&y=pa"
        "&r=no"
        "&chartType=beeswarm"
        "&sort=1"
        "&sortDir=desc"
        "&csv=true"
    )

    response = requests.get(url)
    response.raise_for_status()

    discipline = pd.read_csv(
        StringIO(response.text)
    )

    discipline_columns = [
        "player_id",
        "last_name, first_name",
        "pa",
        "k_percent",
        "bb_percent",
        "whiff_percent",
        "oz_swing_percent",
        "woba",
        "xwoba",
    ]

    discipline = discipline[
        discipline_columns
    ].copy()

    discipline["season"] = season

    discipline = discipline.rename(
        columns={
            "last_name, first_name": "player_name",
            "pa": "plate_appearances",
            "k_percent": "strikeout_pct",
            "bb_percent": "walk_pct",
            "whiff_percent": "whiff_pct",
            "oz_swing_percent": "chase_pct",
        }
    )

    # -------------------------------
    # 3. VALIDATION
    # -------------------------------

    if contact["player_id"].duplicated().any():
        raise ValueError(
            f"Duplicate player IDs found in contact data for {season}"
        )

    if discipline["player_id"].duplicated().any():
        raise ValueError(
            f"Duplicate player IDs found in discipline data for {season}"
        )

    # -------------------------------
    # 4. MERGE
    # -------------------------------

    merged = contact.merge(
        discipline,
        on=["player_id", "season"],
        how="inner",
        suffixes=("_contact", "_discipline"),
    )

    # Keep one player-name field
    merged["player_name"] = merged[
        "player_name_contact"
    ]

    merged = merged.drop(
        columns=[
            "player_name_contact",
            "player_name_discipline",
        ]
    )

    # -------------------------------
    # 5. COLUMN ORDER
    # -------------------------------

    column_order = [
        "player_id",
        "player_name",
        "season",
        "plate_appearances",
        "bbe",

        "walk_pct",
        "strikeout_pct",
        "whiff_pct",
        "chase_pct",

        "avg_exit_velocity",
        "max_exit_velocity",
        "hard_hit_pct",
        "barrel_pct",
        "barrel_per_pa",
        "sweet_spot_pct",
        "avg_launch_angle",

        "woba",
        "xwoba",
    ]

    merged = merged[column_order]

    # -------------------------------
    # 6. FEATURE ENGINEERING
    # -------------------------------

    merged["xwoba_gap"] = (
        merged["xwoba"] - merged["woba"]
    )

    print(
        f"{season}: {len(merged)} hitters collected"
    )

    return merged

In [17]:
test_2024 = collect_hitter_season(2024)

test_2024.head()

2024: 405 hitters collected


,player_id,player_name,season,plate_appearances,bbe,walk_pct,strikeout_pct,whiff_pct,chase_pct,avg_exit_velocity,max_exit_velocity,hard_hit_pct,barrel_pct,barrel_per_pa,sweet_spot_pct,avg_launch_angle,woba,xwoba,xwoba_gap
0,650333,"Arraez, Luis",2024,672,611,3.6,4.3,6.9,35.3,86.3,108.4,23.7,1.6,1.5,40.9,13.7,0.323,0.330,0.007
1,543760,"Semien, Marcus",2024,718,546,8.9,14.6,20.2,25.4,87.4,112.1,35.2,6.6,5.0,33.0,17.0,0.306,0.320,0.014
2,608070,"Ramírez, José",2024,682,544,7.9,12.0,15.2,31.8,89.2,116.6,39.7,8.6,6.9,34.0,19.6,0.364,0.333,-0.031
3,677951,"Witt Jr., Bobby",2024,709,538,8.0,15.0,22.3,32.1,92.7,116.9,48.3,14.3,10.9,35.7,15.1,0.410,0.407,-0.003
4,665489,"Guerrero Jr., Vladimir",2024,697,524,10.3,13.8,21.5,27.5,93.8,117.6,54.9,13.7,10.3,35.1,7.4,0.398,0.413,0.015


In [18]:
print(test_2024.shape)

test_2024[
    [
        "player_name",
        "walk_pct",
        "strikeout_pct",
        "whiff_pct",
        "chase_pct",
        "hard_hit_pct",
        "barrel_pct",
        "woba",
        "xwoba",
        "xwoba_gap",
    ]
].head(10)


#Every season NOT inluding 2020 (Shortened season)
SEASONS = [
    2016,
    2017,
    2018,
    2019,
    2021,
    2022,
    2023,
    2024,
    2025,
    2026,
]

season_datasets = []

for season in SEASONS:
    season_df = collect_hitter_season(season)

    season_datasets.append(season_df)

    season_file = (
        PROCESSED_DATA_DIR
        / f"hitter_features_{season}.csv"
    )

    season_df.to_csv(
        season_file,
        index=False
    )

all_hitter_seasons = pd.concat(
    season_datasets,
    ignore_index=True
)

print(all_hitter_seasons.shape)

all_hitter_seasons.head()

(405, 19)
2016: 391 hitters collected
2017: 387 hitters collected
2018: 390 hitters collected
2019: 406 hitters collected
2021: 404 hitters collected
2022: 411 hitters collected
2023: 403 hitters collected
2024: 405 hitters collected
2025: 389 hitters collected
2026: 406 hitters collected
(3992, 19)


,player_id,player_name,season,plate_appearances,bbe,walk_pct,strikeout_pct,whiff_pct,chase_pct,avg_exit_velocity,max_exit_velocity,hard_hit_pct,barrel_pct,barrel_per_pa,sweet_spot_pct,avg_launch_angle,woba,xwoba,xwoba_gap
0,605141,"Betts, Mookie",2016,730,599,6.7,11.0,14.2,23.3,89.7,114.7,39.9,5.2,4.2,31.7,12.5,0.379,0.338,-0.041
1,514888,"Altuve, Jose",2016,717,580,8.4,9.8,15.3,30.7,88.3,109.0,36.4,6.9,5.6,39.3,11.4,0.391,0.384,-0.007
2,456030,"Pedroia, Dustin",2016,698,564,8.7,10.5,12.2,24.7,88.6,104.1,28.9,2.5,2.0,35.3,8.6,0.358,0.331,-0.027
3,429664,"Canó, Robinson",2016,715,560,6.6,14.0,17.3,34.6,90.1,113.4,42.8,7.3,5.7,32.7,11.5,0.370,0.348,-0.022
4,444876,"Escobar, Alcides",2016,682,556,4.0,14.1,20.6,38.7,84.0,105.8,17.6,1.4,1.2,31.7,9.4,0.278,0.262,-0.016


In [19]:
all_seasons_file = (
    PROCESSED_DATA_DIR
    / "all_hitter_seasons.csv"
)

all_hitter_seasons.to_csv(
    all_seasons_file,
    index=False
)

print(f"Saved combined dataset to: {all_seasons_file}")
print("Shape:", all_hitter_seasons.shape)

Saved combined dataset to: /Users/darekstaudt/Desktop/MLB-breakout-predictor/data/processed/all_hitter_seasons.csv
Shape: (3992, 19)


In [20]:
duplicate_player_seasons = (
    all_hitter_seasons
    .duplicated(
        subset=["player_id", "season"]
    )
    .sum()
)

print(
    "Duplicate player-seasons:",
    duplicate_player_seasons
)

missing_values = (
    all_hitter_seasons
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_values

missing_percent = (
    all_hitter_seasons
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_percent

all_hitter_seasons.info()

all_hitter_seasons[
    "season"
].value_counts().sort_index()

Duplicate player-seasons: 0
<class 'pandas.DataFrame'>
RangeIndex: 3992 entries, 0 to 3991
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   player_id          3992 non-null   int64  
 1   player_name        3992 non-null   str    
 2   season             3992 non-null   int64  
 3   plate_appearances  3992 non-null   int64  
 4   bbe                3992 non-null   int64  
 5   walk_pct           3992 non-null   float64
 6   strikeout_pct      3992 non-null   float64
 7   whiff_pct          3992 non-null   float64
 8   chase_pct          3992 non-null   float64
 9   avg_exit_velocity  3992 non-null   float64
 10  max_exit_velocity  3992 non-null   float64
 11  hard_hit_pct       3992 non-null   float64
 12  barrel_pct         3992 non-null   float64
 13  barrel_per_pa      3992 non-null   float64
 14  sweet_spot_pct     3992 non-null   float64
 15  avg_launch_angle   3992 non-null   float64
 16  woba   

season
2016    391
2017    387
2018    390
2019    406
2021    404
2022    411
2023    403
2024    405
2025    389
2026    406
Name: count, dtype: int64

In [21]:
future_performance = (
    all_hitter_seasons[
        [
            "player_id",
            "season",
            "woba",
            "xwoba"
        ]
    ]
    .copy()
)

future_performance["season"] = (
    future_performance["season"] - 1
)

future_performance = (
    future_performance.rename(
        columns={
            "woba": "next_woba",
            "xwoba": "next_xwoba"
        }
    )
)

future_performance.head()

modeling_data = (
    all_hitter_seasons.merge(
        future_performance,
        on=["player_id", "season"],
        how="left"
    )
)

modeling_data[
    [
        "player_name",
        "season",
        "woba",
        "next_woba"
    ]
].head(20)

modeling_data[
    "woba_change"
] = (
    modeling_data["next_woba"]
    - modeling_data["woba"]
)

modeling_data[
    [
        "player_name",
        "season",
        "woba",
        "next_woba",
        "woba_change"
    ]
].dropna().sort_values(
    "woba_change",
    ascending=False
).head(20)

modeling_data[
    [
        "player_name",
        "season",
        "woba",
        "next_woba",
        "woba_change"
    ]
].dropna().sort_values(
    "woba_change"
).head(20)

training_rows = (
    modeling_data[
        "next_woba"
    ]
    .notna()
    .sum()
)

print(
    "Usable year-to-year observations:",
    training_rows
)

Usable year-to-year observations: 2422


## Defining a Breakout Season

A breakout is defined using year-over-year improvement in wOBA.

Rather than choosing an arbitrary cutoff, the breakout threshold will be based on the historical distribution of wOBA change. Hitters in the top 25% of year-over-year wOBA improvement will initially be classified as breakout seasons.

This definition will be evaluated before modeling to ensure that it produces a meaningful and interpretable target.

In [22]:
training_data = (
    modeling_data
    .dropna(subset=["next_woba"])
    .copy()
)

training_data["woba_change"].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
)

breakout_threshold = (
    training_data["woba_change"]
    .quantile(0.75)
)

print(
    "Breakout threshold:",
    round(breakout_threshold, 3)
)

training_data["breakout"] = (
    training_data["woba_change"]
    >= breakout_threshold
).astype(int)

training_data["breakout"].value_counts()

training_data["breakout"].value_counts(
    normalize=True
).mul(100)

training_data[
    [
        "player_name",
        "season",
        "woba",
        "next_woba",
        "woba_change",
        "breakout"
    ]
].sort_values(
    "woba_change",
    ascending=False
).head(25)

training_data.groupby(
    "breakout"
)[
    [
        "woba",
        "next_woba",
        "woba_change"
    ]
].mean()

training_data[
    [
        "player_name",
        "season",
        "woba",
        "next_woba",
        "woba_change"
    ]
].sort_values(
    "woba_change",
    ascending=False
).head(20)

training_data["big_improvement"] = (
    training_data["woba_change"] >= breakout_threshold
).astype(int)

training_data["current_woba_pct"] = (
    training_data.groupby("season")["woba"]
    .rank(pct=True)
)

training_data["next_season"] = (
    training_data["season"] + 1
)

training_data["next_woba_pct"] = (
    training_data.groupby("next_season")["next_woba"]
    .rank(pct=True)
)

training_data["breakout"] = (
    (training_data["woba_change"] >= breakout_threshold)
    & (training_data["next_woba_pct"] >= 0.60)
).astype(int)

print(
    training_data["breakout"]
    .value_counts()
)

print(
    training_data["breakout"]
    .value_counts(normalize=True)
    .mul(100)
)

training_data[
    [
        "player_name",
        "season",
        "woba",
        "current_woba_pct",
        "next_woba",
        "next_woba_pct",
        "woba_change",
        "breakout"
    ]
].query(
    "breakout == 1"
).sort_values(
    "woba_change",
    ascending=False
).head(30)

for q in [0.75, 0.80, 0.85, 0.90, 0.95]:
    threshold = training_data["woba_change"].quantile(q)

    print(
        f"{int(q * 100)}th percentile:",
        round(threshold, 3)
    )

BREAKOUT_QUANTILE = 0.90

breakout_threshold = (
    training_data["woba_change"]
    .quantile(BREAKOUT_QUANTILE)
)

print(
    "Official breakout threshold:",
    round(breakout_threshold, 3)
)

training_data["breakout"] = (
    (training_data["woba_change"] >= breakout_threshold)
    & (training_data["next_woba_pct"] >= 0.60)
).astype(int)

print(training_data["breakout"].value_counts())

print(
    training_data["breakout"]
    .value_counts(normalize=True)
    .mul(100)
)

breakout_examples = (
    training_data[
        [
            "player_name",
            "season",
            "woba",
            "next_woba",
            "woba_change",
            "current_woba_pct",
            "next_woba_pct",
            "breakout",
        ]
    ]
    .query("breakout == 1")
    .sort_values(
        "woba_change",
        ascending=False
    )
)

breakout_examples.head(40)

season_percentiles = (
    all_hitter_seasons[
        ["player_id", "season", "woba"]
    ]
    .copy()
)

season_percentiles["woba_pct"] = (
    season_percentiles
    .groupby("season")["woba"]
    .rank(pct=True)
)

training_data = training_data.drop(
    columns=[
        "current_woba_pct",
        "next_woba_pct"
    ],
    errors="ignore"
)

training_data = training_data.merge(
    season_percentiles[
        ["player_id", "season", "woba_pct"]
    ],
    on=["player_id", "season"],
    how="left"
)

training_data = training_data.rename(
    columns={
        "woba_pct": "current_woba_pct"
    }
)

next_percentiles = (
    season_percentiles[
        ["player_id", "season", "woba_pct"]
    ]
    .copy()
)

next_percentiles["season"] = (
    next_percentiles["season"] - 1
)

next_percentiles = next_percentiles.rename(
    columns={
        "woba_pct": "next_woba_pct"
    }
)

training_data = training_data.merge(
    next_percentiles,
    on=["player_id", "season"],
    how="left"
)

BREAKOUT_QUANTILE = 0.90

breakout_threshold = (
    training_data["woba_change"]
    .quantile(BREAKOUT_QUANTILE)
)

training_data["breakout"] = (
    (training_data["woba_change"] >= breakout_threshold)
    & (training_data["next_woba_pct"] >= 0.60)
).astype(int)

print(
    "Official breakout threshold:",
    round(breakout_threshold, 3)
)

print(
    training_data["breakout"]
    .value_counts()
)

print(
    training_data["breakout"]
    .value_counts(normalize=True)
    .mul(100)
)

modeling_data.to_csv(
    PROCESSED_DATA_DIR / "modeling_data.csv",
    index=False
)

training_data.to_csv(
    PROCESSED_DATA_DIR / "training_data.csv",
    index=False
)

print("Modeling datasets saved.")

Breakout threshold: 0.021


breakout
0    2013
1     409
Name: count, dtype: int64
breakout
0    83.11313
1    16.88687
Name: proportion, dtype: float64
75th percentile: 0.021
80th percentile: 0.027
85th percentile: 0.034
90th percentile: 0.045
95th percentile: 0.063
Official breakout threshold: 0.045
breakout
0    2233
1     189
Name: count, dtype: int64
breakout
0    92.196532
1     7.803468
Name: proportion, dtype: float64
Official breakout threshold: 0.045
breakout
0    2229
1     193
Name: count, dtype: int64
breakout
0    92.031379
1     7.968621
Name: proportion, dtype: float64
Modeling datasets saved.


### Breakout Definition

A breakout season is defined as a year-over-year increase in wOBA at or above the 90th percentile of historical improvement.

In the training sample, this corresponds to an improvement of approximately 0.045 wOBA.

To prevent large improvements from very low offensive baselines from being classified as breakouts, the hitter must also finish the following season at or above the 60th percentile in wOBA.

This definition allows both emerging players and established players making major performance jumps to qualify as breakouts.

## Summary

This notebook created the multi-season hitter dataset used throughout the project. Data from Statcast and Baseball Savant were cleaned, standardized, and merged using MLB player IDs.

Eligible hitter-seasons were then matched with the following season's performance, producing **2,422 year-to-year observations** for exploratory analysis and predictive modeling.